# Dataset Analysis - Allographs and Sign Variations

This notebook performs exploratory data analysis on the Indus script dataset,
focusing on allographic variety (identical signs based on engraving style) as shown in Figure 59.

The analysis ensures the AI recognizes that "messy handwriting" on pot sherds doesn't change the underlying alphabetic sign.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from pathlib import Path
import pandas as pd
from collections import defaultdict

# Set paths
project_root = Path().absolute().parent
data_dir = project_root / "data" / "processed" / "train" / "primary_core_signs"

print(f"Project root: {project_root}")
print(f"Data directory: {data_dir}")

## 1. Load and Analyze Dataset Structure

In [ ]:
# Get all sign classes
sign_classes = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
print(f"Total sign classes: {len(sign_classes)}")
print(f"\nSign classes:")
for i, sign_class in enumerate(sign_classes, 1):
    print(f"  {i}. {sign_class}")

In [ ]:
# Count images per class
class_counts = {}
for sign_class in sign_classes:
    class_dir = data_dir / sign_class
    image_files = list(class_dir.glob("*.png")) + list(class_dir.glob("*.jpg"))
    class_counts[sign_class] = len(image_files)

# Create DataFrame for analysis
df_counts = pd.DataFrame(list(class_counts.items()), columns=['Sign_Class', 'Image_Count'])
df_counts = df_counts.sort_values('Image_Count', ascending=False)

print("\nImage counts per class:")
print(df_counts.to_string(index=False))

## 2. Visualize Class Distribution

In [ ]:
plt.figure(figsize=(15, 8))
plt.bar(range(len(df_counts)), df_counts['Image_Count'])
plt.xlabel('Sign Class')
plt.ylabel('Number of Images')
plt.title('Distribution of Images Across Sign Classes')
plt.xticks(range(len(df_counts)), df_counts['Sign_Class'], rotation=90, fontsize=8)
plt.tight_layout()
plt.show()

print(f"\nTotal images in dataset: {df_counts['Image_Count'].sum()}")
print(f"Average images per class: {df_counts['Image_Count'].mean():.2f}")
print(f"Min images per class: {df_counts['Image_Count'].min()}")
print(f"Max images per class: {df_counts['Image_Count'].max()}")

## 3. Analyze Allographic Variations (Figure 59)

This section analyzes variations in sign appearance based on engraving style,
material, and artistic interpretation while maintaining the same underlying sign.

In [ ]:
def load_images_from_class(class_name, max_images=10):
    """Load sample images from a specific sign class"""
    class_dir = data_dir / class_name
    image_files = list(class_dir.glob("*.png")) + list(class_dir.glob("*.jpg"))
    images = []
    
    for img_file in image_files[:max_images]:
        img = cv2.imread(str(img_file), cv2.IMREAD_GRAYSCALE)
        if img is not None:
            images.append(img)
    
    return images

def analyze_image_variations(images):
    """Analyze variations in a set of images of the same sign"""
    if len(images) == 0:
        return None
    
    # Resize all to same size for comparison
    target_size = (64, 64)
    resized_images = [cv2.resize(img, target_size) for img in images]
    
    # Calculate statistics
    mean_intensities = [np.mean(img) for img in resized_images]
    std_intensities = [np.std(img) for img in resized_images]
    pixel_densities = [np.sum(img > 0) / (64 * 64) for img in resized_images]
    
    return {
        'mean_intensity': np.mean(mean_intensities),
        'std_intensity': np.std(mean_intensities),
        'mean_std': np.mean(std_intensities),
        'mean_density': np.mean(pixel_densities),
        'density_variation': np.std(pixel_densities)
    }

# Analyze variations for each class
class_variations = {}
for sign_class in sign_classes:
    images = load_images_from_class(sign_class)
    variations = analyze_image_variations(images)
    if variations:
        class_variations[sign_class] = variations

# Create DataFrame for variations
df_variations = pd.DataFrame.from_dict(class_variations, orient='index')
print("Allographic Variation Analysis:")
print(df_variations.head(10))

## 4. Visualize Sample Images from Each Class

In [ ]:
def visualize_class_samples(class_names, samples_per_class=3):
    """Visualize sample images from multiple classes"""
    num_classes = len(class_names)
    fig, axes = plt.subplots(num_classes, samples_per_class, figsize=(12, 3*num_classes))
    
    if num_classes == 1:
        axes = axes.reshape(1, -1)
    
    for row, class_name in enumerate(class_names):
        images = load_images_from_class(class_name, max_images=samples_per_class)
        
        for col in range(samples_per_class):
            ax = axes[row, col]
            if col < len(images):
                ax.imshow(images[col], cmap='gray')
            ax.axis('off')
            if col == 0:
                ax.set_ylabel(class_name, fontsize=10)
    
    plt.tight_layout()
    plt.show()

# Visualize first 10 classes
print("Sample images from first 10 sign classes:")
visualize_class_samples(sign_classes[:10], samples_per_class=3)

## 5. Identify Classes with High Allographic Variation

In [ ]:
# Sort classes by density variation (measure of allographic variety)
df_variations_sorted = df_variations.sort_values('density_variation', ascending=False)

print("Classes with highest allographic variation:")
print(df_variations_sorted.head(10))

# Visualize top variable classes
top_variable_classes = df_variations_sorted.head(5).index.tolist()
print(f"\nTop 5 classes with highest variation: {top_variable_classes}")

## 6. Statistical Analysis of Sign Characteristics

In [ ]:
# Correlation analysis
correlation_matrix = df_variations.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Sign Characteristics')
plt.tight_layout()
plt.show()

# Distribution analysis
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].hist(df_variations['mean_intensity'], bins=20)
axes[0, 0].set_title('Distribution of Mean Intensity')
axes[0, 0].set_xlabel('Mean Intensity')

axes[0, 1].hist(df_variations['mean_std'], bins=20)
axes[0, 1].set_title('Distribution of Standard Deviation')
axes[0, 1].set_xlabel('Standard Deviation')

axes[1, 0].hist(df_variations['mean_density'], bins=20)
axes[1, 0].set_title('Distribution of Pixel Density')
axes[1, 0].set_xlabel('Pixel Density')

axes[1, 1].hist(df_variations['density_variation'], bins=20)
axes[1, 1].set_title('Distribution of Density Variation')
axes[1, 1].set_xlabel('Density Variation')

plt.tight_layout()
plt.show()

## 7. Recommendations for Data Augmentation

Based on the allographic analysis, recommend data augmentation strategies to help the CNN recognize sign variations.

In [ ]:
print("=" * 60)
print("DATA AUGMENTATION RECOMMENDATIONS")
print("=" * 60)

# Classes with few samples (need more augmentation)
low_sample_classes = df_counts[df_counts['Image_Count'] < 5]['Sign_Class'].tolist()
print(f"\n1. Classes with low sample count (<5 images): {len(low_sample_classes)}")
for cls in low_sample_classes:
    print(f"   - {cls}")

# Classes with high variation (need robust augmentation)
high_variation_classes = df_variations_sorted.head(10).index.tolist()
print(f"\n2. Classes with high allographic variation: {len(high_variation_classes)}")
for cls in high_variation_classes:
    print(f"   - {cls}")

print("\n3. Recommended augmentation techniques:")
print("   - Random rotation (±15°) to account for engraving angles")
print("   - Random scaling (0.9-1.1) for size variations")
print("   - Elastic transformations for stroke variations")
print("   - Random contrast adjustments for material differences")
print("   - Gaussian noise for surface texture variations")

print("\n4. Special attention needed for:")
print("   - Signs with similar structures (e.g., P341 Oval vs P341 Leaf)")
print("   - Compound signs that may be decomposed")
print("   - Signs with high density variation")

## 8. Export Analysis Results

In [ ]:
# Save analysis results
output_dir = project_root / "notebooks" / "analysis_results"
output_dir.mkdir(exist_ok=True)

# Save class counts
df_counts.to_csv(output_dir / "class_counts.csv", index=False)

# Save variation analysis
df_variations.to_csv(output_dir / "allographic_variations.csv")

# Save recommendations
with open(output_dir / "augmentation_recommendations.txt", 'w') as f:
    f.write("Data Augmentation Recommendations\n")
    f.write("=" * 40 + "\n\n")
    f.write(f"Low sample classes: {low_sample_classes}\n")
    f.write(f"High variation classes: {high_variation_classes}\n")

print(f"\nAnalysis results saved to: {output_dir}")

## Summary

This analysis provides:
1. **Dataset Overview**: Distribution of images across 40 primary core signs
2. **Allographic Analysis**: Understanding of natural variations in sign appearance
3. **Variation Metrics**: Quantitative measures of sign complexity and variability
4. **Augmentation Strategy**: Data augmentation recommendations to improve model robustness
5. **Quality Assessment**: Identification of classes needing more samples or attention

The analysis ensures the CNN will be trained to recognize the underlying alphabetic sign despite variations in engraving style, material, and artistic interpretation - crucial for matching Keeladi graffiti to the Indus alphabet.